# Assignment 4: DHF Attack Evaluation
This notebook evaluates the Diversifying the High-level Features (DHF) attack.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys

REPO_DIR = '/content/transferattacktInterns'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', 'pgn-attack', 'https://github.com/Chidroopakanaparthy/transferattack.git', REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Repo set up at', REPO_DIR)

In [ ]:
%cd /content/transferattacktInterns
!pip install deepface pandas numpy pillow tf-keras -q
!gdown 1CD1NjufQJeImCMjZbI_yDDeWWMmRVvR_
!unzip -q dataset_extractedfaces.zip

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from deepface import DeepFace
from core.transfer_attack_core import run_attack

ATTACKERS = ['Facenet512', 'ArcFace', 'GhostFaceNet', 'VGG-Face']
VICTIMS = ['Facenet512', 'ArcFace', 'GhostFaceNet', 'VGG-Face']
ATTACKER_INPUT_SIZES = {'Facenet512': (160, 160), 'ArcFace': (112, 112), 'GhostFaceNet': (112, 112), 'VGG-Face': (224, 224)}
VICTIM_INPUT_SIZES = {'Facenet512': (160, 160), 'ArcFace': (112, 112), 'GhostFaceNet': (112, 112), 'VGG-Face': (224, 224)}

with open('core/verification_thresholds.json') as f:
    THRESHOLDS = json.load(f)

subset_csv = "docs/subset_input_pairs.csv"
dataset_root = "dataset_extractedfaces"
df = pd.read_csv(subset_csv)

def load_img(path: str, size: tuple) -> np.ndarray:
    img = Image.open(path).convert('RGB').resize(size)
    arr = np.array(img).astype('float32') / 255.0
    arr = (arr - 0.5) * 2.0
    return arr[None]

raw_rows = []

for attacker_name in ATTACKERS:
    print(f'\nEvaluating attacker: {attacker_name}')
    input_size = ATTACKER_INPUT_SIZES[attacker_name]
    attacker_model = DeepFace.build_model(attacker_name).model
    
    victim_models = {}
    for vname in VICTIMS:
        if attacker_name == vname: continue
        victim_models[vname] = DeepFace.build_model(vname).model

    for idx, row in df.iterrows():
        row_id = int(row['row_id'])
        dataset = str(row['dataset'])
        attack_type = str(row['attack_type'])
        
        img_path = str(row['img1'])
        src_path = os.path.join(dataset_root, img_path.split('dataset_extractedfaces/', 1)[-1] if 'dataset_extractedfaces/' in img_path else img_path.lstrip('/'))
        img_path2 = str(row['img2'])
        tgt_path = os.path.join(dataset_root, img_path2.split('dataset_extractedfaces/', 1)[-1] if 'dataset_extractedfaces/' in img_path2 else img_path2.lstrip('/'))
            
        src_arr = load_img(src_path, input_size)
        tgt_arr = load_img(tgt_path, input_size)
        src_t = tf.constant(src_arr, dtype=tf.float32)
        tgt_t = tf.constant(tgt_arr, dtype=tf.float32)
        
        adv_t = run_attack('DHF', attacker_model, src_t, tgt_t, attack_type, input_size)
        adv_arr = adv_t.numpy()
        
        for vname, vmodel in victim_models.items():
            v_size = VICTIM_INPUT_SIZES[vname]
            arr_v_adv = tf.image.resize(tf.constant(adv_arr), v_size).numpy()
            out_adv = vmodel(tf.constant(arr_v_adv), training=False)
            if isinstance(out_adv, list): out_adv = out_adv[0]
            adv_emb_v = tf.nn.l2_normalize(out_adv, axis=1).numpy()
            
            arr_v_tgt = tf.image.resize(tf.constant(tgt_arr), v_size).numpy()
            out_tgt = vmodel(tf.constant(arr_v_tgt), training=False)
            if isinstance(out_tgt, list): out_tgt = out_tgt[0]
            tgt_emb_v = tf.nn.l2_normalize(out_tgt, axis=1).numpy()
            
            adv_sim = float(np.sum(adv_emb_v * tgt_emb_v))
            
            raw_rows.append({
                'row_id': row_id,
                'attacker_model': attacker_name,
                'victim_model': vname,
                'dataset': dataset,
                'attack_type': attack_type,
                'adv_sim': adv_sim,
                'threshold': THRESHOLDS[vname][dataset]['threshold']
            })

res_df = pd.DataFrame(raw_rows)
def get_breach(row):
    if row['attack_type'] == 'impersonation_attack':
        return 1 if row['adv_sim'] >= row['threshold'] else 0
    else:
        return 1 if row['adv_sim'] < row['threshold'] else 0
res_df['breach'] = res_df.apply(get_breach, axis=1)

print("\n--- DHF Evaluation Summary ---")
print(f"Total evaluations: {len(res_df)}")
print(f"Overall Breach Rate: {res_df['breach'].mean() * 100:.2f}%")

print("\n--- By Goal ---")
by_goal = res_df.groupby('attack_type')['breach'].mean() * 100
print(by_goal)